# 12 - Heterogeneity-aware FL (FedPer & clustered) lintas-dataset (Paper 4, poin reviewer B-7)

**Tujuan:** menjawab kelemahan terbesar yang disebut reviewer: *"Anda menunjukkan
FedAvg gagal, tapi belum menunjukkan apakah metode FL yang SADAR-HETEROGENITAS
bisa mengatasi kegagalan itu."* Kita uji dua strategi standar pada kasus
cross-dataset CIC<->UNSW yang kolaps di nb03:

1. **FedPer (personalisasi):** layer awal (shared backbone) di-FedAvg; layer
   akhir (personalized head) DISIMPAN LOKAL per-klien -> tiap klien punya kepala
   sendiri yang menyesuaikan prior lokal. Evaluasi: tiap test set pakai kepala
   klien sumbernya; GLOBAL = gabungan prediksi per-domain.
2. **Clustered-FL (K=1 per klien):** batas atas personalisasi -> tiap klien model
   penuh sendiri (identik local-only, dilaporkan sbg pembanding sadar-heterogenitas).

**Tujuan ilmiah (JUJUR):** bukan mencari metode terbaik, tetapi menunjukkan bahwa
**failure boundary yang kami identifikasi dapat sebagian dimitigasi** ketika
strategi federasi memperhitungkan heterogenitas klien. Bandingkan vs FedAvg naif
(nb03 GLOBAL -0.026) dan local-only (0.614).

**Konsistensi:** MLP 9->128->64->1 (sama dgn nb03). z-score lokal. Hyperparameter
sama (ROUNDS=100, LOCAL_EPOCHS=5, LR=3e-3). Output: `fed_out/personalized.csv`.

> Jalankan setelah nb01. JANGAN mengarang angka.

## 0. Setup + MLP numpy dgn pemisahan backbone/head (untuk FedPer)

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import (matthews_corrcoef, recall_score, precision_score,
                             f1_score, balanced_accuracy_score)
OUTDIR='fed_out'; os.makedirs(OUTDIR, exist_ok=True)
ROUNDS=100; LOCAL_EPOCHS=5; LR=3e-3; SEED=42
LAYERS=[9,128,64,1]  # identik nb03/fed_node.py

def _relu(x): return np.maximum(0,x)
def _sig(x): return 1.0/(1.0+np.exp(-np.clip(x,-30,30)))
def init_w(seed=SEED):
    r=np.random.RandomState(seed); W=[]
    for a,b in zip(LAYERS[:-1],LAYERS[1:]):
        W.append(r.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W
def forward(W,X):
    a1=X@W[0]+W[1]; z1=_relu(a1)
    a2=z1@W[2]+W[3]; z2=_relu(a2)
    a3=z2@W[4]+W[5]; out=_sig(a3).ravel()
    return out,(X,a1,z1,a2,z2)
def train_local(W,X,y,epochs=LOCAL_EPOCHS,lr=LR,batch=256,seed=SEED):
    r=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=r.permutation(n)
        for s in range(0,n,batch):
            bi=idx[s:s+batch]; Xb=X[bi]; yb=y[bi].astype(float)
            out,(X_,a1,z1,a2,z2)=forward(W,Xb)
            g=(out-yb)/len(yb)
            gW4=z2.T@g[:,None]; gb4=g.sum(0)
            d2=(g[:,None]@W[4].T)*(a2>0); gW2=z1.T@d2; gb2=d2.sum(0)
            d1=(d2@W[2].T)*(a1>0); gW0=X_.T@d1; gb0=d1.sum(0)
            gall=[gW0,gb0,gW2,gb2,gW4,gb4]
            for i,gv in enumerate(gall): W[i]=W[i]-lr*np.asarray(gv).reshape(W[i].shape)
    return W
def fedavg(ws,sizes,idxs=None):
    # agregasi HANYA pada indeks bobot di 'idxs' (untuk FedPer: hanya backbone)
    tot=float(sum(sizes)); out=[w.copy() for w in ws[0]]
    rng=range(len(ws[0])) if idxs is None else idxs
    for i in rng:
        out[i]=sum((sizes[k]/tot)*ws[k][i] for k in range(len(ws)))
    return out
def zfit(X): mu=X.mean(0); sd=X.std(0); sd[sd==0]=1.0; return mu,sd
def zap(X,mu,sd): return (X-mu)/sd
print('MLP + FedPer helper siap | LAYERS',LAYERS)

## 1. Muat partisi lintas-dataset (dari nb01)

In [ ]:
def load(name):
    p=os.path.join(OUTDIR,name)
    if not os.path.exists(p): print('MISSING',p); return None,None
    d=np.load(p,allow_pickle=True); return np.asarray(d['X'],float), np.asarray(d['y'],int)
Xc_tr,yc_tr=load('cic_train.npz'); Xc_te,yc_te=load('cic_test.npz')
Xu_tr,yu_tr=load('unsw_train.npz'); Xu_te,yu_te=load('unsw_test.npz')
ok=all(v is not None for v in [Xc_tr,Xu_tr,Xc_te,Xu_te])
if ok:
    Xg_te=np.vstack([Xc_te,Xu_te]); yg_te=np.concatenate([yc_te,yu_te])
    print('CIC',Xc_tr.shape,'UNSW',Xu_tr.shape,'GLOBAL-test',Xg_te.shape)
else:
    print('(jalankan nb01 dulu)')

## 2. Helper evaluasi

In [ ]:
def metrics(y,yp):
    return dict(MCC=round(matthews_corrcoef(y,yp),4),
                recall=round(recall_score(y,yp,zero_division=0),4),
                precision=round(precision_score(y,yp,zero_division=0),4),
                F1=round(f1_score(y,yp,zero_division=0),4),
                bal_acc=round(balanced_accuracy_score(y,yp),4))
def predict(W,X,mu,sd):
    out,_=forward(W,zap(X,mu,sd)); return (out>=0.5).astype(int)
res=[]

## 3. FedPer: backbone di-FedAvg, head (W[4],W[5]) LOKAL per-klien

Indeks bobot: W[0],W[1]=layer1; W[2],W[3]=layer2; W[4],W[5]=head output.
FedPer: agregasi hanya indeks {0,1,2,3} (backbone); {4,5} tetap personal.

In [ ]:
if ok:
    clients={'cic':(Xc_tr,yc_tr),'unsw':(Xu_tr,yu_tr)}
    zc={cid:zfit(X) for cid,(X,y) in clients.items()}
    Xz={cid:zap(clients[cid][0],*zc[cid]) for cid in clients}
    sizes=[len(clients['cic'][1]),len(clients['unsw'][1])]
    BACKBONE=[0,1,2,3]  # agregasi; head {4,5} personal
    Wg=init_w(SEED)
    Wloc={cid:[w.copy() for w in Wg] for cid in clients}  # tiap klien simpan head sendiri
    curve=[]
    for t in range(ROUNDS):
        updated={}
        for cid in clients:
            # mulai dari backbone global + head lokal klien
            Wstart=[Wg[i] if i in BACKBONE else Wloc[cid][i] for i in range(6)]
            Wk=train_local(Wstart, Xz[cid], clients[cid][1])
            updated[cid]=Wk
            Wloc[cid]=[Wk[i].copy() for i in range(6)]  # simpan head terbaru
        # FedAvg HANYA backbone
        Wg=fedavg([updated['cic'],updated['unsw']], sizes, idxs=BACKBONE)
        # monitoring: MCC per-domain pakai head lokal masing2, lalu gabung
        def pred_dom(cid,Xte):
            Wp=[Wg[i] if i in BACKBONE else Wloc[cid][i] for i in range(6)]
            return predict(Wp,Xte,*zc[cid])
        ypc=pred_dom('cic',Xc_te); ypu=pred_dom('unsw',Xu_te)
        yp_glob=np.concatenate([ypc,ypu])
        curve.append({'round':t,'MCC_global':round(matthews_corrcoef(yg_te,yp_glob),4)})
    # evaluasi akhir FedPer
    def final_pred(cid,Xte): 
        Wp=[Wg[i] if i in BACKBONE else Wloc[cid][i] for i in range(6)]
        return predict(Wp,Xte,*zc[cid])
    ypc=final_pred('cic',Xc_te); ypu=final_pred('unsw',Xu_te)
    res.append({'setting':'FedPer','eval':'CIC-test', **metrics(yc_te,ypc)})
    res.append({'setting':'FedPer','eval':'UNSW-test', **metrics(yu_te,ypu)})
    res.append({'setting':'FedPer','eval':'GLOBAL-test', **metrics(yg_te,np.concatenate([ypc,ypu]))})
    print('FedPer selesai')
print('=== SEL 3 (FedPer) SELESAI ===')

## 4. Clustered-FL (tiap klien model penuh sendiri = batas atas personalisasi)

Setara local-only tapi dibingkai sbg strategi sadar-heterogenitas (tiap klien
cluster sendiri). Evaluasi tiap test pakai model klien sumbernya; GLOBAL gabung.

In [ ]:
if ok:
    models={}
    for cid,(X,y) in clients.items():
        mu,sd=zc[cid]; W=init_w(SEED)
        W=train_local(W, zap(X,mu,sd), y, epochs=ROUNDS*LOCAL_EPOCHS)  # setara total update
        models[cid]=W
    ypc=predict(models['cic'],Xc_te,*zc['cic'])
    ypu=predict(models['unsw'],Xu_te,*zc['unsw'])
    res.append({'setting':'Clustered(per-client)','eval':'CIC-test', **metrics(yc_te,ypc)})
    res.append({'setting':'Clustered(per-client)','eval':'UNSW-test', **metrics(yu_te,ypu)})
    res.append({'setting':'Clustered(per-client)','eval':'GLOBAL-test', **metrics(yg_te,np.concatenate([ypc,ypu]))})
    print('clustered selesai')
print('=== SEL 4 (clustered) SELESAI ===')

## 5. Simpan + banding dgn FedAvg naif (nb03) & local-only

**Kunci:** apakah FedPer / clustered GLOBAL-test MCC >> FedAvg naif (-0.026)?
Jika ya -> failure boundary dapat SEBAGIAN dimitigasi oleh strategi
sadar-heterogenitas (pesan untuk reviewer poin B-7).

In [ ]:
if ok and len(res):
    dfres=pd.DataFrame(res); dfres.to_csv(os.path.join(OUTDIR,'personalized.csv'),index=False)
    pd.DataFrame(curve).to_csv(os.path.join(OUTDIR,'personalized_curve.csv'),index=False)
    import IPython.display as ipd; ipd.display(dfres)
    g=[r for r in res if r['eval']=='GLOBAL-test']
    print('\nGLOBAL-test MCC:')
    for r in g: print(f"  {r['setting']}: {r['MCC']}")
    print('  (pembanding) FedAvg naif nb03: -0.026 | local-only: 0.614')
else:
    print('(lewati: jalankan nb01 dulu)')
print('=== SEL 5 (hasil + banding) SELESAI ===')

## 6. Unggah ke S3 (results/personalized)

In [ ]:
S3_BUCKET=os.environ.get('FED_S3_BUCKET','ssh-detection-features-232032302717')
REGION=os.environ.get('AWS_REGION','ap-southeast-1'); PREFIX='unsw-far/federated/results/personalized'
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for f in ('personalized.csv','personalized_curve.csv'):
        p=os.path.join(OUTDIR,f)
        if os.path.exists(p): s3.upload_file(p,S3_BUCKET,f'{PREFIX}/{f}'); up+=1; print('  ok ->',f)
    print(f'\nunggah {up} -> s3://{S3_BUCKET}/{PREFIX}/')
except Exception as e: print('(upload S3 gagal:',e,')')
print('=== SEL 6 (unggah) SELESAI ===')

## 7. Catatan untuk naskah (poin reviewer B-7)

- Tambah baris FedPer & Clustered ke tabel cross-dataset -> tunjukkan apakah
  strategi sadar-heterogenitas memitigasi kegagalan FedAvg naif.
- Narasi JUJUR: FedPer berbagi backbone tapi personalisasi head -> jika GLOBAL
  MCC naik jauh di atas -0.026, itu bukti 'failure boundary dapat sebagian
  dimitigasi'. Clustered = batas atas (tiap klien model sendiri).
- Evaluasi FedPer/clustered memakai head/model klien SUMBER tiap test (deployment
  realistis: tiap node punya kepala personalisasi sendiri). Dilaporkan apa adanya.
- Jika FedPer TIDAK menolong juga -> tetap temuan jujur (personalisasi head saja
  tak cukup; perlu alignment distribusi -> future work).